# Exploratory Sentiment Analysis

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Output directories
figures_dir = Path("../results/figures/sentiment")
tables_dir = Path("../results/tables/sentiment")

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

data_adidas = pd.read_excel("../data/input/adidas.xlsx", sheet_name="Sheet1")
data_puma = pd.read_excel("../data/input/puma.xlsx", sheet_name="Sheet1")
data_asics = pd.read_excel("../data/input/asics.xlsx", sheet_name="Sheet1")
data_mizuno = pd.read_excel("../data/input/mizuno.xlsx", sheet_name="Sheet1")

datasets = {
    "adidas": data_adidas,
    "puma": data_puma,
    "asics": data_asics,
    "mizuno": data_mizuno,
}

# Model labels and colors
model_names = {
    "financial_roberta": "Financial-RoBERTa",
    "finbert": "FinBERT"
}

model_colors = {
    "financial_roberta": "#527DA5",
    "finbert": "#D88A48"
}

def save_sentiment_figure(fig, company, name):
    """Save sentiment figures as vector PDF files."""

    fig.savefig(
        figures_dir / f"{company}_{name}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

### 3.1 Distribution of Sentiment Scores

This analysis examines the distribution of continuous linguistic sentiment scores estimated by Financial-RoBERTa and FinBERT.

Sentiment scores range from −1 to +1, with negative values indicating negative linguistic sentiment and positive values indicating positive linguistic sentiment.

The distributions are examined separately for each company and model. Descriptive statistics are calculated to summarize the central tendency and dispersion of sentiment scores.

In [ ]:
bins = np.linspace(-1, 1, 31)

for company, df in results.items():

    rows = []

    fig, axes = plt.subplots(
        1, 2,
        figsize=(10.5, 4),
        sharex=True,
        sharey=True
    )

    for ax, (model_key, model_label) in zip(
        axes, model_names.items()
    ):

        scores = df[f"{model_key}_score"].dropna()

        # Descriptive statistics
        rows.append({
            "model": model_label,
            "n": len(scores),
            "mean": scores.mean(),
            "median": scores.median(),
            "std": scores.std(),
            "q25": scores.quantile(0.25),
            "q75": scores.quantile(0.75)
        })

        # Histogram
        ax.hist(
            scores,
            bins=bins,
            color=model_colors[model_key],
            edgecolor="white"
        )

        # Mean sentiment score
        if len(scores):
            ax.axvline(
                scores.mean(),
                color="black",
                linestyle="--",
                linewidth=1,
                label="Mean"
            )

            ax.legend(frameon=False)

        ax.set_title(model_label)
        ax.set_xlabel("Linguistic sentiment score")
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)

    axes[0].set_ylabel("Number of segments")
    axes[0].set_xlim(-1, 1)

    fig.suptitle(
        f"Sentiment Score Distribution — {company.capitalize()}"
    )

    fig.tight_layout()

    save_sentiment_figure(
        fig, company, "score_distribution"
    )

    # Save descriptive statistics
    pd.DataFrame(rows).round(3).to_csv(
        tables_dir / f"{company}_score_summary.csv",
        index=False
    )

### 3.2 Sentiment by Leadership Context and Information Valence

This analysis examines how linguistic sentiment varies across the manually classified Leadership Contexts and Information Valence categories.

The median sentiment score and interquartile range are used to describe the distribution of sentiment within each category.

Information Valence represents the favorability of the underlying information, whereas linguistic sentiment captures the evaluative orientation expressed through language.

The analysis provides descriptive evidence relevant to the subsequent regression models, without controlling for other explanatory variables.

In [ ]:
context_order = [
    "PERF", "STRAT", "MARKET",
    "CHALL", "PEOPLE", "STAKE", "OTHER"
]

valence_order = ["negative", "neutral", "positive"]


def group_summary(df, group_column, order):

    labels = df[group_column].astype("string").str.strip()

    if group_column == "Leadership_context":
        labels = labels.str.upper()
    else:
        labels = labels.str.lower()

    rows = []

    for group in order:

        mask = labels.eq(group).fillna(False)

        for model_key, model_label in model_names.items():

            scores = df.loc[
                mask, f"{model_key}_score"
            ].dropna()

            rows.append({
                "category": group,
                "model": model_label,
                "n": len(scores),
                "mean": scores.mean(),
                "median": scores.median(),
                "q25": scores.quantile(0.25),
                "q75": scores.quantile(0.75)
            })

    return pd.DataFrame(rows)


for company, df in results.items():

    # Calculate category summaries
    context_summary = group_summary(
        df, "Leadership_context", context_order
    )

    valence_summary = group_summary(
        df, "Information_valence", valence_order
    )

    # Save tables
    context_summary.round(3).to_csv(
        tables_dir / f"{company}_sentiment_by_context.csv",
        index=False
    )

    valence_summary.round(3).to_csv(
        tables_dir / f"{company}_sentiment_by_valence.csv",
        index=False
    )

    # Figure
    fig, axes = plt.subplots(
        1, 2,
        figsize=(13, 4.5),
        gridspec_kw={"width_ratios": [1.6, 1]}
    )

    panels = [
        (
            axes[0], context_summary,
            context_order, "Leadership Context"
        ),
        (
            axes[1], valence_summary,
            valence_order, "Information Valence"
        )
    ]

    for ax, summary, order, title in panels:

        positions = np.arange(len(order))

        for i, (model_key, model_label) in enumerate(
            model_names.items()
        ):

            tab = (
                summary.loc[summary["model"].eq(model_label)]
                .set_index("category")
                .reindex(order)
            )

            median = tab["median"]
            q25 = tab["q25"]
            q75 = tab["q75"]

            valid = median.notna()

            x = positions[valid] + (
                -0.12 if i == 0 else 0.12
            )

            # Interquartile range
            yerr = np.vstack([
                (median[valid] - q25[valid]).to_numpy(),
                (q75[valid] - median[valid]).to_numpy()
            ])

            ax.errorbar(
                x,
                median[valid],
                yerr=yerr,
                fmt="o",
                color=model_colors[model_key],
                capsize=3,
                label=model_label
            )

        ax.set_xticks(
            positions, order,
            rotation=35, ha="right"
        )

        ax.set_title(title)
        ax.set_ylim(-1.05, 1.05)
        ax.axhline(0, color="gray", linewidth=0.8)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)

    axes[0].set_ylabel("Median sentiment score (IQR)")

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False
    )

    fig.suptitle(
        f"Sentiment by Context and Valence — {company.capitalize()}"
    )

    fig.tight_layout(rect=(0, 0.08, 1, 0.94))

    save_sentiment_figure(
        fig, company, "context_valence_scores"
    )

### 3.3 Temporal Distribution of Sentiment

This analysis examines descriptive changes in average linguistic sentiment across the reporting period from 2020 to 2025.

Annual mean sentiment scores are calculated separately for each company and sentiment model.

The analysis identifies potential temporal patterns in corporate leadership communication without attributing observed changes to specific organisational or external events.

In [ ]:
years = list(range(2020, 2026))

for company, df in results.items():

    rows = []

    fig, ax = plt.subplots(figsize=(8, 4.3))

    year_numeric = pd.to_numeric(
        df["Year"], errors="coerce"
    )

    for model_key, model_label in model_names.items():

        summary = (
            pd.DataFrame({
                "year": year_numeric,
                "score": df[f"{model_key}_score"]
            })
            .groupby("year")["score"]
            .agg(
                n="count",
                mean="mean",
                median="median"
            )
            .reindex(years)
        )

        # Annual mean sentiment
        ax.plot(
            years,
            summary["mean"],
            marker="o",
            linewidth=1.8,
            color=model_colors[model_key],
            label=model_label
        )

        for year, row in summary.iterrows():

            rows.append({
                "year": year,
                "model": model_label,
                "n": int(row["n"]) if pd.notna(row["n"]) else 0,
                "mean": row["mean"],
                "median": row["median"]
            })

    ax.set_xticks(years)
    ax.set_ylim(-1, 1)

    ax.set_ylabel("Mean linguistic sentiment score")
    ax.set_xlabel("Reporting year")

    ax.set_title(
        f"Sentiment by Year — {company.capitalize()}"
    )

    ax.axhline(
        0,
        color="gray",
        linewidth=0.8
    )

    ax.grid(axis="y", alpha=0.2)
    ax.legend(frameon=False)

    fig.tight_layout()

    save_sentiment_figure(
        fig, company, "sentiment_by_year"
    )

    # Save annual descriptive statistics
    pd.DataFrame(rows).round(3).to_csv(
        tables_dir / f"{company}_sentiment_by_year.csv",
        index=False
    )